In [15]:
# My Modeul
from my_package.select_dataset_all import get_all_dataframe_from_database

# External Package
import pandas as pd

# Learning
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

#### 함수

In [16]:
def determine_label(row):
    if row['STEEP_LABEL'] == 1:
        return 1
    elif row['SLOWLY_LABEL'] == 1:
        return 2
    elif row['OUT_OF_WATER_STEEP'] == 1:
        return 3
    elif row['HUNTING'] == 1:
        return 4
    elif row['TIME_OFFSET'] == 1:
        return 5
    else:
        return 0

In [17]:
def train_xgboost_multiple_model(data):
    
    """
    불균형 데이터셋에서 XGBoost 모델을 사용하여 불량 탐지를 수행하는 함수입니다.

    Parameters:
    - X: Feature 데이터 (DataFrame 또는 numpy array)
    - y: Label 데이터 (불량 라벨, Series 또는 numpy array)

    Returns:
    - model: 훈련된 XGBoost 모델
    - evaluation: 모델 평가 결과 (dict 형식)
    """
    

    # 1. 데이터셋 로드
    X = data.drop(columns = ['label']).values  # 특징 (features)
    y = data['label'].values.ravel()  # 레이블 (labels)

    # 2. 학습 데이터와 테스트 데이터로 분리
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # 클래스 가중치 계산
    classes = sorted(set(y))
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}
    
    
  # XGBoost 모델 설정
    model = xgb.XGBClassifier(
        objective='multi:softmax', # 다중 클래스 분류 문제
        num_class=len(classes),  # 클래스 개수
        max_depth=6,
        learning_rate=0.1,
        n_estimators=100,
        eval_metric='mlogloss',  # 다중 클래스 로그 손실
        scale_pos_weight=class_weights,
        use_label_encoder=False,
        random_state=42
    )
    # 4. 모델 학습
    model.fit(X_train, y_train)

    # 5. 예측 수행
    y_pred = model.predict(X_test)

    evaluation = {
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "classification_report": classification_report(y_test, y_pred)
    }
    

    print(f"evaluation: {evaluation}")

#### 스크립트 실행

In [18]:
# goals: 데이터 로드

tro = get_all_dataframe_from_database('tc_ai_fault_group_up','ecs_test') 

In [19]:
# goals: 함수 적용

# 변수 변경
data = tro[['CSU', 'STS', 'FTS', 'FMU', 'CURRENT', 'TRO', 'TRO_DIFF', 'PEAK_VALLEY_INDICES', 'CROSS_CORRELATION','RE_CROSS_CORRELATION','TRO_NEG_COUNT','RE_CROSS_CORRELATION_COUNT',
                 'STEEP_LABEL', 'SLOWLY_LABEL','OUT_OF_WATER_STEEP', 'HUNTING', 'TIME_OFFSET']]

# apply 함수로 각 행에 대해 determine_label 함수를 적용하여 'label' 열 생성
data['label'] = data.apply(determine_label, axis=1)

#data = data[['CSU', 'STS', 'FTS', 'FMU', 'CURRENT', 'TRO', 'TRO_DIFF', 'PEAK_VALLEY_INDICES', 'CROSS_CORRELATION', 'RE_CROSS_CORRELATION','TRO_NEG_COUNT','RE_CROSS_CORRELATION_COUNT','label']]

C:\Users\pc021\AppData\Local\Temp\ipykernel_1868\4087442213.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['label'] = data.apply(determine_label, axis=1)


In [20]:
# goals: 모델 학습

train_xgboost_multiple_model(data)

evaluation: {'confusion_matrix': array([[613,   0,   0,   0,   0,   0],
       [  0,  36,   0,   0,   0,   0],
       [  0,   0,   5,   0,   0,   0],
       [  0,   0,   0,  71,   0,   0],
       [  0,   0,   1,   0,  37,   0],
       [  0,   0,   0,   0,   0,  62]], dtype=int64), 'classification_report': '              precision    recall  f1-score   support\n\n           0       1.00      1.00      1.00       613\n           1       1.00      1.00      1.00        36\n           2       0.83      1.00      0.91         5\n           3       1.00      1.00      1.00        71\n           4       1.00      0.97      0.99        38\n           5       1.00      1.00      1.00        62\n\n    accuracy                           1.00       825\n   macro avg       0.97      1.00      0.98       825\nweighted avg       1.00      1.00      1.00       825\n'}


C:\Users\pc021\Desktop\프로젝트\테크로스\모델 고도화\모델\model\lib\site-packages\xgboost\core.py:158: UserWarning: [14:44:53] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [17]:
data[data['TIME_OFFSET']!=0]

,CSU,STS,FTS,FMU,CURRENT,TRO,TRO_DIFF,PEAK_VALLEY_INDICES,CROSS_CORRELATION,RE_CROSS_CORRELATION,TRO_NEG_COUNT,RE_CROSS_CORRELATION_COUNT,STEEP_LABEL,SLOWLY_LABEL,OUT_OF_WATER_STEEP,HUNTING,TIME_OFFSET,label
0,24.300744,0.000000,38.880579,1073.205868,8001.487603,6.517769,0.072397,6,3.367359,1746.888815,34,4,0,0,0,0,1,5
2,23.233550,0.000000,37.285976,881.560296,6674.230769,6.550828,0.054083,9,3.039436,2712.280610,55,7,0,0,0,0,2,0
3,23.999179,0.000000,37.719254,968.061866,7165.805970,6.355746,0.033657,7,3.534500,2045.962574,67,5,0,0,0,0,1,5
4,22.688736,0.000000,40.579011,824.495220,6567.862637,6.772198,0.035495,10,3.617816,2935.801750,121,8,0,0,0,1,2,4
139,53.307628,0.000000,0.000000,697.890657,4161.781022,7.485438,0.013850,7,6.017646,10394.950054,160,5,1,0,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2306,36.432842,18.418889,26.758295,2073.443333,13344.757106,6.645013,0.000000,23,4.929808,4837.191695,16,4,0,0,11,3,1,5
2307,36.432842,18.418889,26.758295,2073.443333,13344.757106,6.645013,0.000000,23,4.929808,4837.191695,16,4,0,0,11,3,1,5
2372,35.213566,10.846434,32.896949,1991.047868,13083.702206,8.430404,0.061507,28,3.425159,2214.442026,90,5,7,1,1,3,1,2
2529,33.749959,13.546986,18.021120,290.461161,899.233198,3.650061,0.001354,10,-2.274656,2683.738226,22,5,0,0,0,1,1,4
